In [ ]:
#pip install pyspark==3.5.1

## Creating a SparkSession

- A **SparkSession** object is the entry point to Spark.  
- It **wraps the Driver program** (the "brain" of your Spark app).  
- Every Spark command you run (like `spark.read.csv(...)` or `df.groupBy(...)`) goes through this `spark` object.  

### Example
```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .getOrCreate()


In [1]:
# --- Setup SparkSession (local mode) ---
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Beginner-DE-Demo")
    .master("local[*]")         # use all local cores
    .getOrCreate()
)

spark


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/11 14:06:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In local mode:

There is no external cluster manager.

Spark just launches executor threads inside your own process.

[*] means → “use as many threads as CPU cores available.”

Memory allocation is just whatever your JVM (Java Virtual Machine) gets.


## 1. Read CSV into a DataFrame
- Use `spark.read.csv()` to load raw data.
- Can provide a schema or let Spark infer it.
- Always check the row count, schema, and preview the data.


In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Path to the CSV DATASET
csv_path = "Data/retail_events.csv"

In [3]:
csv_path

'Data/retail_events.csv'

In [4]:
#create a dataframe, similar to pandas but we have more POWER
df_raw = (
    spark.read
    .option("header", True)
    .csv(csv_path)
)


In [5]:
df_raw.show(5)

+-------------------+-------+----------+-------+-------+------+-----+--------+----------+--------------------+
|         event_time|user_id|product_id|country| device|source|price|quantity|promo_code|               email|
+-------------------+-------+----------+-------+-------+------+-----+--------+----------+--------------------+
|2025-08-10T15:06:00|   1007|       204|     US|android| email|66.92|       1|      NULL|user1007@example.com|
|2025-08-10T12:10:00|   1006|       200|     DE|    web|   ads|28.79|       2|  SUMMER25|user1006@example.com|
|2025-08-10T10:24:00|   1007|       201|     US|    web|direct|70.94|       2|      NULL| USER1007@EXAMPLE...|
|2025-08-10T15:21:00|   1005|       203|     US|    ios|   seo|66.87|       2|      NULL|user1005@example.com|
|2025-08-10T09:17:00|   1003|       204|     CA|    ios|   ads|27.83|       1|      NULL|user1003@example.com|
+-------------------+-------+----------+-------+-------+------+-----+--------+----------+--------------------+
o

In [6]:
# (Optional) Provide a schema so types are correct from the start
#WE DO THIS TO RENAME COLUMNS, SET DATA TYPES AKA MORE CONTROL
schema = StructType([
    StructField("event_time", StringType(), True),   # we'll cast to timestamp later
    StructField("user_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("country", StringType(), True),
    StructField("device", StringType(), True),
    StructField("source", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("promo_code", StringType(), True),
    StructField("email", StringType(), True),
])



df_raw = (
    spark.read
    .option("header", True)
    .schema(schema)     # comment this OUT to infer schema automatically
    .csv(csv_path)
)


In [7]:
df_raw.show(5)

+-------------------+-------+----------+-------+-------+------+-----+--------+----------+--------------------+
|         event_time|user_id|product_id|country| device|source|price|quantity|promo_code|               email|
+-------------------+-------+----------+-------+-------+------+-----+--------+----------+--------------------+
|2025-08-10T15:06:00|   1007|       204|     US|android| email|66.92|       1|      NULL|user1007@example.com|
|2025-08-10T12:10:00|   1006|       200|     DE|    web|   ads|28.79|       2|  SUMMER25|user1006@example.com|
|2025-08-10T10:24:00|   1007|       201|     US|    web|direct|70.94|       2|      NULL| USER1007@EXAMPLE...|
|2025-08-10T15:21:00|   1005|       203|     US|    ios|   seo|66.87|       2|      NULL|user1005@example.com|
|2025-08-10T09:17:00|   1003|       204|     CA|    ios|   ads|27.83|       1|      NULL|user1003@example.com|
+-------------------+-------+----------+-------+-------+------+-----+--------+----------+--------------------+
o

In [8]:
#count rows
df_raw.count()

41

In [9]:
#quickly view the data
df_raw.show(5, truncate=False)

+-------------------+-------+----------+-------+-------+------+-----+--------+----------+----------------------+
|event_time         |user_id|product_id|country|device |source|price|quantity|promo_code|email                 |
+-------------------+-------+----------+-------+-------+------+-----+--------+----------+----------------------+
|2025-08-10T15:06:00|1007   |204       |US     |android|email |66.92|1       |NULL      |user1007@example.com  |
|2025-08-10T12:10:00|1006   |200       |DE     |web    |ads   |28.79|2       |SUMMER25  |user1006@example.com  |
|2025-08-10T10:24:00|1007   |201       |US     |web    |direct|70.94|2       |NULL      | USER1007@EXAMPLE.COM |
|2025-08-10T15:21:00|1005   |203       |US     |ios    |seo   |66.87|2       |NULL      |user1005@example.com  |
|2025-08-10T09:17:00|1003   |204       |CA     |ios    |ads   |27.83|1       |NULL      |user1003@example.com  |
+-------------------+-------+----------+-------+-------+------+-----+--------+----------+-------

In [10]:
#we can see our col names, data types and rules
df_raw.printSchema()

root
 |-- event_time: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- source: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- email: string (nullable = true)



## 2. Basic Cleaning
- Cast columns (e.g., string → timestamp)
- Normalize strings (trim, lowercase/uppercase)
- Handle nulls and empty values
- Drop duplicates
- Filter out invalid data (e.g., quantity > 0)

In [11]:
from pyspark.sql import functions as F

#lets copy our dataframe
df = df_raw


In [12]:
#Cast event_time to timestamp
df = df.withColumn("event_ts", F.to_timestamp("event_time"))

In [14]:
df.show(5, truncate=False)

+-------------------+-------+----------+-------+-------+------+-----+--------+----------+----------------------+-------------------+
|event_time         |user_id|product_id|country|device |source|price|quantity|promo_code|email                 |event_ts           |
+-------------------+-------+----------+-------+-------+------+-----+--------+----------+----------------------+-------------------+
|2025-08-10T15:06:00|1007   |204       |US     |android|email |66.92|1       |NULL      |user1007@example.com  |2025-08-10 15:06:00|
|2025-08-10T12:10:00|1006   |200       |DE     |web    |ads   |28.79|2       |SUMMER25  |user1006@example.com  |2025-08-10 12:10:00|
|2025-08-10T10:24:00|1007   |201       |US     |web    |direct|70.94|2       |NULL      | USER1007@EXAMPLE.COM |2025-08-10 10:24:00|
|2025-08-10T15:21:00|1005   |203       |US     |ios    |seo   |66.87|2       |NULL      |user1005@example.com  |2025-08-10 15:21:00|
|2025-08-10T09:17:00|1003   |204       |CA     |ios    |ads   |27.83|

In [15]:
#trim() removes extra spaces at the beginning and end of a string.
#normalize string columns (lowercase emails, devices, sources, etc.)
df = (
    df
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("device", F.lower(F.trim(F.col("device"))))
    .withColumn("source", F.lower(F.trim(F.col("source"))))
    .withColumn("promo_code", F.upper(F.trim(F.col("promo_code"))))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
)

In [16]:
df.show(5)

+-------------------+-------+----------+-------+-------+------+-----+--------+----------+--------------------+-------------------+
|         event_time|user_id|product_id|country| device|source|price|quantity|promo_code|               email|           event_ts|
+-------------------+-------+----------+-------+-------+------+-----+--------+----------+--------------------+-------------------+
|2025-08-10T15:06:00|   1007|       204|     US|android| email|66.92|       1|      NULL|user1007@example.com|2025-08-10 15:06:00|
|2025-08-10T12:10:00|   1006|       200|     DE|    web|   ads|28.79|       2|  SUMMER25|user1006@example.com|2025-08-10 12:10:00|
|2025-08-10T10:24:00|   1007|       201|     US|    web|direct|70.94|       2|      NULL|user1007@example.com|2025-08-10 10:24:00|
|2025-08-10T15:21:00|   1005|       203|     US|    ios|   seo|66.87|       2|      NULL|user1005@example.com|2025-08-10 15:21:00|
|2025-08-10T09:17:00|   1003|       204|     CA|    ios|   ads|27.83|       1|     

In [17]:
# Replace empty strings with nulls for selected columns
for c in ["country", "promo_code"]:
    df = df.withColumn(c, F.when(F.col(c) == "", F.lit(None)).otherwise(F.col(c)))


In [18]:
df.show(5)

+-------------------+-------+----------+-------+-------+------+-----+--------+----------+--------------------+-------------------+
|         event_time|user_id|product_id|country| device|source|price|quantity|promo_code|               email|           event_ts|
+-------------------+-------+----------+-------+-------+------+-----+--------+----------+--------------------+-------------------+
|2025-08-10T15:06:00|   1007|       204|     US|android| email|66.92|       1|      NULL|user1007@example.com|2025-08-10 15:06:00|
|2025-08-10T12:10:00|   1006|       200|     DE|    web|   ads|28.79|       2|  SUMMER25|user1006@example.com|2025-08-10 12:10:00|
|2025-08-10T10:24:00|   1007|       201|     US|    web|direct|70.94|       2|      NULL|user1007@example.com|2025-08-10 10:24:00|
|2025-08-10T15:21:00|   1005|       203|     US|    ios|   seo|66.87|       2|      NULL|user1005@example.com|2025-08-10 15:21:00|
|2025-08-10T09:17:00|   1003|       204|     CA|    ios|   ads|27.83|       1|     

In [19]:

# Drop exact duplicate rows (on all columns)
df = df.dropDuplicates()


In [20]:
# Simple data-quality filters (business rules)
#    - quantity should be >= 1 (remove zero or negative)
#    - price should be > 0
df = df.filter((F.col("quantity") >= 1) & (F.col("price") > 0))  #we lose rows


In [21]:
df.count()

36

In [22]:
# Add derived columns (total line amount, event_date)
df = df.withColumn("line_amount", F.col("price") * F.col("quantity"))
df = df.withColumn("event_date", F.to_date("event_ts"))


In [23]:
df.select("event_time", "event_ts", "user_id", "product_id", "country", "price", "quantity", "line_amount").show(5, truncate=False)

+-------------------+-------------------+-------+----------+-------+-----+--------+-----------+
|event_time         |event_ts           |user_id|product_id|country|price|quantity|line_amount|
+-------------------+-------------------+-------+----------+-------+-----+--------+-----------+
|2025-08-10T08:11:00|2025-08-10 08:11:00|1004   |201       |US     |90.09|3       |270.27     |
|2025-08-10T09:52:00|2025-08-10 09:52:00|1007   |204       |DE     |63.62|1       |63.62      |
|2025-08-10T12:59:00|2025-08-10 12:59:00|1000   |201       |US     |100.1|2       |200.2      |
|2025-08-10T17:24:00|2025-08-10 17:24:00|1009   |202       |IN     |5.48 |1       |5.48       |
|2025-08-10T16:32:00|2025-08-10 16:32:00|1008   |205       |IN     |90.25|1       |90.25      |
+-------------------+-------------------+-------+----------+-------+-----+--------+-----------+
only showing top 5 rows



## 3. Basic Analysis
- Count rows and distinct values
- Check for nulls in each column
- Get descriptive statistics for numeric columns

In [24]:
# Count rows
print("Cleaned row count:", df.count())


Cleaned row count: 36


In [25]:
# Distinct users/products/countries
distinct_stats = df.agg(
    F.countDistinct("user_id").alias("unique_users"),
    F.countDistinct("product_id").alias("unique_products"),
    F.countDistinct("country").alias("unique_countries")
)
distinct_stats.show()


+------------+---------------+----------------+
|unique_users|unique_products|unique_countries|
+------------+---------------+----------------+
|           9|              6|               5|
+------------+---------------+----------------+



In [26]:
# Simple missing/null counts per column
null_counts = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show(truncate=False)


+----------+-------+----------+-------+------+------+-----+--------+----------+-----+--------+-----------+----------+
|event_time|user_id|product_id|country|device|source|price|quantity|promo_code|email|event_ts|line_amount|event_date|
+----------+-------+----------+-------+------+------+-----+--------+----------+-----+--------+-----------+----------+
|0         |0      |0         |1      |0     |0     |0    |0       |21        |0    |0       |0          |0         |
+----------+-------+----------+-------+------+------+-----+--------+----------+-----+--------+-----------+----------+



In [27]:
# Quick descriptive stats on numeric columns
df.select("price", "quantity", "line_amount").describe().show()

+-------+-----------------+------------------+-----------------+
|summary|            price|          quantity|      line_amount|
+-------+-----------------+------------------+-----------------+
|  count|               36|                36|               36|
|   mean|58.60888888888888|              1.75|107.2472222222222|
| stddev|34.48984237861327|0.7699721701835353|86.18967938253101|
|    min|             5.48|                 1|             5.48|
|    max|           117.48|                 3|           352.44|
+-------+-----------------+------------------+-----------------+



## 4. Basic Aggregations
- Revenue by country
- Top products by sales/revenue
- Daily revenue trend

In [28]:

rev_by_country = (
    df.groupBy("country")
      .agg(
          F.count("*").alias("rows"),
          F.sum("line_amount").alias("revenue"),
          F.countDistinct("user_id").alias("unique_users")
      )
      .orderBy(F.desc("revenue"))
)
rev_by_country.show(truncate=False)


+-------+----+-----------------+------------+
|country|rows|revenue          |unique_users|
+-------+----+-----------------+------------+
|US     |14  |1683.23          |9           |
|GB     |5   |628.9499999999999|3           |
|CA     |6   |615.32           |5           |
|DE     |3   |473.64           |2           |
|IN     |7   |409.56           |5           |
|NULL   |1   |50.2             |1           |
+-------+----+-----------------+------------+



In [29]:
# Top products by revenue
top_products = (
    df.groupBy("product_id")
      .agg(
          F.sum("quantity").alias("units_sold"),
          F.sum("line_amount").alias("revenue")
      )
      .orderBy(F.desc("revenue"))
)
top_products.show(truncate=False)



+----------+----------+-----------------+
|product_id|units_sold|revenue          |
+----------+----------+-----------------+
|201       |18        |998.74           |
|204       |11        |941.95           |
|203       |16        |752.4900000000001|
|205       |8         |728.6199999999999|
|200       |6         |303.86           |
|202       |4         |135.24           |
+----------+----------+-----------------+



In [30]:
# c) Daily revenue trend
daily_rev = (
    df.groupBy("event_date")
      .agg(
          F.sum("line_amount").alias("revenue"),
          F.countDistinct("user_id").alias("unique_users")
      )
      .orderBy("event_date")
)
daily_rev.show(truncate=False)

+----------+------------------+------------+
|event_date|revenue           |unique_users|
+----------+------------------+------------+
|2025-08-10|3860.8999999999996|9           |
+----------+------------------+------------+



## 5. Write Cleaned Data
- Save cleaned data as CSV `df.write.format("<file_format>").save("<output_path>")`
- Save in Parquet format (efficient for analytics)
- Partition by country/date for faster queries

In [31]:
# Choose output folders (local paths for notebook demo)
out_base = "Data/retail_events_out"




In [32]:
# Write CSV (coalesce to 1 file for easy inspection in a demo)
(
    df.coalesce(1)
      .write
      .mode("overwrite")
      .option("header", True)
      .csv(f"{out_base}/csv_cleaned")
)



In [33]:
# Write Parquet (good default for analytics)
(
    df.write
      .mode("overwrite")
      .parquet(f"{out_base}/parquet_cleaned")
)

print("Wrote cleaned CSV to:", f"{out_base}/csv_cleaned")
print("Wrote cleaned Parquet to:", f"{out_base}/parquet_cleaned")

[Stage 75:>                                                         (0 + 1) / 1]

Wrote cleaned CSV to: Data/retail_events_out/csv_cleaned
Wrote cleaned Parquet to: Data/retail_events_out/parquet_cleaned
